Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from tempfile import mkdtemp
import shutil

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV,StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report,accuracy_score

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)
from scipy.stats import randint,loguniform

from string import punctuation



In [ ]:
df=pd.read_excel("icthub_dataset.xlsx")
df.drop_duplicates(inplace=True)
print(df['Category'].value_counts())
df.head()

Asking for Internship Details         82
General_Info_Services                 68
Thanking Message                      68
AI Internship Details                 66
Cyber Security Internship Details     64
Web Development internship details    64
data analysis internship details      64
flutter internship details            64
supply chain internship details       64
Greeting                              59
Location                              50
Name: Category, dtype: int64


,User Input,Category,Chatbot Response
0,Good morning,Greeting,Welcome to ICTHUB - For Digital Solutions! \n\...
1,Good afternoon,Greeting,Welcome to ICTHUB - For Digital Solutions! \n\...
2,Good evening,Greeting,Welcome to ICTHUB - For Digital Solutions! \n...
3,Hi there,Greeting,\n Welcome to ICTHUB - For Digital Solutions!...
4,Hello,Greeting,\n Welcome to ICTHUB - For Digital Solutions!...


Data Preprocessing

In [ ]:
def preprocessing_text(text):
  text=text.lower()
  text=word_tokenize(text)
  text=[word for word in text if word.isalnum()]

  stop_words= set(stopwords.words('english'))
  text=[word for word in text if word not in stop_words and word not in punctuation]


  stemmer=PorterStemmer()
  text=[stemmer.stem(word) for word in text]
  
  return " ".join(text)

In [12]:
df['processed_text'] = df['User Input'].apply(preprocessing_text)
df.head()

,User Input,Category,Chatbot Response,processed_text
0,Good morning,Greeting,Welcome to ICTHUB - For Digital Solutions! \n\...,good morn
1,Good afternoon,Greeting,Welcome to ICTHUB - For Digital Solutions! \n\...,good afternoon
2,Good evening,Greeting,Welcome to ICTHUB - For Digital Solutions! \n...,good even
3,Hi there,Greeting,\n Welcome to ICTHUB - For Digital Solutions!...,hi
4,Hello,Greeting,\n Welcome to ICTHUB - For Digital Solutions!...,hello


In [13]:
x = df['processed_text']
y = df['Category']

label_encoder = LabelEncoder()
encoded_y = label_encoder.fit_transform(y)


X_train, X_test, y_train, y_test = train_test_split(x,
                                                    encoded_y,
                                                    test_size=0.3,
                                                    random_state=42,
                                                    stratify=encoded_y)


print("Categories ")

for index,label in enumerate(df['Category'].unique()):
    print(f"{index} : {label}")



Categories 
0 : Greeting
1 : General_Info_Services
2 : Location
3 : Asking for Internship Details
4 : AI Internship Details
5 : Cyber Security Internship Details
6 : Web Development internship details
7 : data analysis internship details
8 : flutter internship details
9 : supply chain internship details
10 : Thanking Message


Training

In [14]:
cv = StratifiedKFold(n_splits=10,shuffle=True,random_state=42)
cache = mkdtemp()
pipe = Pipeline(
    steps= [('Vectorizer',TfidfVectorizer()),('model',LogisticRegression(random_state=42))],
    memory = cache
)
param = [
      {
      'model__C':loguniform(1e-5,1e0),
      'model__max_iter':randint(100,1000),
     
      }   
]


random_model = RandomizedSearchCV(
    pipe,
    param,
    cv=cv,
    scoring='accuracy',
    refit = True,
    random_state=42,
    n_iter=100,
    n_jobs=-1,
    verbose=3
)
try:
    random_model.fit(X_train,y_train)
    print(f"train Score : {random_model.best_score_*100:.2f}%")
    
    results = pd.DataFrame(random_model.cv_results_)
    coloms = ['rank_test_score','mean_test_score']
    final_table = results[coloms].sort_values(by='rank_test_score')
    
    print(final_table.head(10).to_string(index=False))
    print("=============================================")
    
    y_pred = random_model.predict(X_test)
    print(f"test Score : {accuracy_score(y_test,y_pred)*100:.2f}%")
    
    print("--------------------")
    print(random_model.best_params_)
    print("--------------------")
    print("Classification Report : \n",classification_report(y_test,y_pred))
    
finally:
    shutil.rmtree(cache)  
best_model = random_model.best_estimator_

Fitting 10 folds for each of 100 candidates, totalling 1000 fits
train Score : 95.18%
 rank_test_score  mean_test_score
               1         0.951796
               1         0.951796
               3         0.947796
               4         0.941796
               5         0.935796
               5         0.935796
               7         0.917755
               7         0.917755
               9         0.897714
              10         0.877673
test Score : 95.79%
--------------------
{'model__C': 0.6732248920775336, 'model__max_iter': 341}
--------------------
Classification Report : 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       0.93      1.00      0.96        25
           2       1.00      1.00      1.00        19
           3       0.91      1.00      0.95        20
           4       0.81      0.94      0.87        18
           5       1.00      0.80      0.89        15
           6    

Save

In [15]:

#joblib.dump(best_model, "model.pkl")
#joblib.dump(label_encoder, "label_encoder.pkl")